# Notebook 30

# Experiment 2

## Probability Threshold Optimization for Random Forest

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
train_dataset = pd.read_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/train_dataset_v5_lane_vehicle.csv"
)

test_dataset = pd.read_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/test_dataset_v5_lane_vehicle.csv"
)

print(train_dataset.shape)
print(test_dataset.shape)

(242965, 174)
(66060, 174)


In [4]:
behavior_map = {
    "NORMAL1": "NORMAL",
    "NORMAL2": "NORMAL"
}

train_dataset["behavior"] = train_dataset["behavior"].replace(behavior_map)
test_dataset["behavior"] = test_dataset["behavior"].replace(behavior_map)

print(train_dataset["behavior"].value_counts())

print()

print(test_dataset["behavior"].value_counts())

behavior
NORMAL        116203
AGGRESSIVE     63761
DROWSY         63001
Name: count, dtype: int64

behavior
DROWSY        36143
AGGRESSIVE    15257
NORMAL        14660
Name: count, dtype: int64


In [5]:
ttc_columns = [
    "ttc_mean",
    "ttc_std",
    "ttc_variance",
    "ttc_min",
    "ttc_max",
    "ttc_median",
    "ttc_rms",
    "ttc_skewness",
    "ttc_kurtosis",
    "ttc_q25",
    "ttc_q75",
    "ttc_iqr"
]

for col in ttc_columns:

    if col in train_dataset.columns:

        if any(x in col for x in [
            "std",
            "variance",
            "iqr",
            "skewness",
            "kurtosis"
        ]):

            train_dataset[col] = train_dataset[col].fillna(0)
            test_dataset[col] = test_dataset[col].fillna(0)

        else:

            train_dataset[col] = train_dataset[col].fillna(100)
            test_dataset[col] = test_dataset[col].fillna(100)

train_dataset = train_dataset.fillna(0)
test_dataset = test_dataset.fillna(0)

print(train_dataset.isnull().sum().sum())
print(test_dataset.isnull().sum().sum())

0
0


In [6]:
target_column = "behavior"

feature_columns = [

    col

    for col in train_dataset.columns

    if col not in [

        "behavior",
        "driver",
        "trip",
        "road_type"

    ]

]

X_train = train_dataset[feature_columns]

X_test = test_dataset[feature_columns]

y_train = train_dataset[target_column]

y_test = test_dataset[target_column]

print(X_train.shape)
print(X_test.shape)

(242965, 170)
(66060, 170)


In [7]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

print(label_encoder.classes_)

['AGGRESSIVE' 'DROWSY' 'NORMAL']


In [8]:
rf_model = RandomForestClassifier(

    n_estimators=500,

    random_state=42,

    n_jobs=-1

)

rf_model.fit(
    X_train,
    y_train
)

print("Baseline Random Forest trained successfully!")

Baseline Random Forest trained successfully!


In [10]:
rf_pred = rf_model.predict(X_test)

rf_pred[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [11]:
from sklearn.metrics import accuracy_score

rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

print(f"Random Forest Accuracy : {rf_accuracy:.4f}")

Random Forest Accuracy : 0.6302


# Threshold Search

The probability threshold of the DROWSY class will be systematically adjusted to analyze its impact on recall, precision, and overall accuracy.

In [12]:
rf_prob = rf_model.predict_proba(X_test)

print(rf_prob.shape)

print(label_encoder.classes_)

(66060, 3)
['AGGRESSIVE' 'DROWSY' 'NORMAL']


In [13]:
def evaluate_threshold(threshold):

    predictions = []

    for probs in rf_prob:

        if probs[1] >= threshold:
            predictions.append(1)
        else:
            predictions.append(np.argmax(probs))

    predictions = np.array(predictions)

    report = classification_report(
        y_test,
        predictions,
        target_names=label_encoder.classes_,
        output_dict=True
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    return {

        "Threshold": threshold,

        "Accuracy": accuracy,

        "Drowsy Recall": report["DROWSY"]["recall"],

        "Drowsy Precision": report["DROWSY"]["precision"],

        "Normal Precision": report["NORMAL"]["precision"]

    }

In [14]:
threshold_results = []

for threshold in np.arange(0.30, 0.61, 0.05):

    result = evaluate_threshold(
        round(threshold, 2)
    )

    threshold_results.append(result)

threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results

,Threshold,Accuracy,Drowsy Recall,Drowsy Precision,Normal Precision
0,0.30,0.677157,0.718313,0.756402,0.404684
1,0.35,0.661777,0.644910,0.782102,0.393408
2,0.40,0.644641,0.585923,0.798138,0.381750
3,0.45,0.634650,0.547713,0.811478,0.378146
4,0.50,0.630170,0.533990,0.814724,0.375820
5,0.55,0.630170,0.533990,0.814724,0.375820
6,0.60,0.630170,0.533990,0.814724,0.375820


In [15]:
threshold_results.sort_values(
    by="Drowsy Recall",
    ascending=False
)

,Threshold,Accuracy,Drowsy Recall,Drowsy Precision,Normal Precision
0,0.30,0.677157,0.718313,0.756402,0.404684
1,0.35,0.661777,0.644910,0.782102,0.393408
2,0.40,0.644641,0.585923,0.798138,0.381750
3,0.45,0.634650,0.547713,0.811478,0.378146
4,0.50,0.630170,0.533990,0.814724,0.375820
5,0.55,0.630170,0.533990,0.814724,0.375820
6,0.60,0.630170,0.533990,0.814724,0.375820


In [16]:
print(label_encoder.classes_)

print()

print(rf_model.classes_)

['AGGRESSIVE' 'DROWSY' 'NORMAL']

[0 1 2]


In [18]:
threshold = 0.30

rf_pred_threshold = []

for probs in rf_prob:

    if probs[1] >= threshold:
        rf_pred_threshold.append(1)
    else:
        rf_pred_threshold.append(np.argmax(probs))

rf_pred_threshold = np.array(rf_pred_threshold)

In [19]:
baseline_pred = rf_model.predict(X_test)

print(np.mean(baseline_pred == rf_pred_threshold))

0.8390251286709053


In [20]:
print("Baseline Accuracy")

print(
    accuracy_score(
        y_test,
        rf_model.predict(X_test)
    )
)

print()

print("Threshold Accuracy")

print(
    accuracy_score(
        y_test,
        rf_pred_threshold
    )
)

Baseline Accuracy
0.6301695428398426

Threshold Accuracy
0.6771571298819256


In [21]:
print(classification_report(
    y_test,
    rf_model.predict(X_test),
    target_names=label_encoder.classes_
))

              precision    recall  f1-score   support

  AGGRESSIVE       0.85      0.75      0.80     15257
      DROWSY       0.81      0.53      0.65     36143
      NORMAL       0.38      0.74      0.50     14660

    accuracy                           0.63     66060
   macro avg       0.68      0.68      0.65     66060
weighted avg       0.72      0.63      0.65     66060



# Experiment 2B

## Relative Threshold Optimization

Objective

Instead of only using an absolute probability threshold, compare the DROWSY probability with the NORMAL probability to reduce unnecessary DROWSY predictions.

In [23]:
def evaluate_relative_threshold(threshold, ratio):

    predictions = []

    for probs in rf_prob:

        drowsy_prob = probs[1]
        normal_prob = probs[2]

        if (
            drowsy_prob >= threshold and
            drowsy_prob >= normal_prob * ratio
        ):
            predictions.append(1)

        else:
            predictions.append(np.argmax(probs))

    predictions = np.array(predictions)

    report = classification_report(
        y_test,
        predictions,
        target_names=label_encoder.classes_,
        output_dict=True
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    return {

        "Threshold": threshold,

        "Ratio": ratio,

        "Accuracy": accuracy,

        "Drowsy Recall": report["DROWSY"]["recall"],

        "Drowsy Precision": report["DROWSY"]["precision"],

        "Normal Precision": report["NORMAL"]["precision"]

    }

In [24]:
relative_results = []

for ratio in [0.80, 0.85, 0.90, 0.95, 1.00]:

    result = evaluate_relative_threshold(
        threshold=0.30,
        ratio=ratio
    )

    relative_results.append(result)

relative_results = pd.DataFrame(relative_results)

relative_results

,Threshold,Ratio,Accuracy,Drowsy Recall,Drowsy Precision,Normal Precision
0,0.3,0.80,0.649803,0.610104,0.783980,0.388861
1,0.3,0.85,0.645913,0.596187,0.788640,0.386326
2,0.3,0.90,0.640902,0.580278,0.793320,0.382633
3,0.3,0.95,0.635180,0.563456,0.797096,0.378806
4,0.3,1.00,0.630503,0.548488,0.802104,0.375820


In [25]:
relative_results.sort_values(
    by="Drowsy Recall",
    ascending=False
)

,Threshold,Ratio,Accuracy,Drowsy Recall,Drowsy Precision,Normal Precision
0,0.3,0.80,0.649803,0.610104,0.783980,0.388861
1,0.3,0.85,0.645913,0.596187,0.788640,0.386326
2,0.3,0.90,0.640902,0.580278,0.793320,0.382633
3,0.3,0.95,0.635180,0.563456,0.797096,0.378806
4,0.3,1.00,0.630503,0.548488,0.802104,0.375820


## Experiment 2 Conclusion

Two threshold optimization strategies were evaluated.

### Absolute Threshold

- Produced the highest DROWSY recall.
- Improved overall accuracy.
- Improved NORMAL precision.

### Relative Threshold

- Produced more conservative predictions.
- Reduced DROWSY recall.
- Did not outperform the absolute threshold.

Therefore, the absolute threshold strategy (threshold = 0.30) was selected for subsequent experiments.